In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch.nn.functional as F
from torch.nn import Parameter
from torchinfo import summary
from torchvision import transforms, datasets, models
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import math
import os
import random
import numpy as np
from termcolor import cprint, colored
from tqdm.auto import tqdm
from PIL import Image
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
CONFIGURATION = {
    'train_dir': './Dataset/Normal_Map/train',
    'test_dir': './Dataset/Normal_Map/test',
    'epochs': 3000,
    'num_workers': 4,
    'batch_size': 16,
    'image_size': 224,
    'num_class': len(os.listdir('./Dataset/Normal_Map/train')),
    'embedding_size': 512,
    
    'learning_rate': 0.1,
    'weight_decay': 5e-4,
    'momentum': 0.9,
    'alpha': 0.9,
    
    # Hàm m(ai) giúp thay đổi ai từ 0.25 đến 1.6
    'scale': 64,
    'lambda_g': 20,
    'l_margin': 0.45, 
    'u_margin': 0.8,
    'l_a': 10, 
    'u_a': 110,
}


# Dataloader

In [3]:
# Đặt seed toàn cục
seed = 42
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

In [4]:
def fixed_image_standardization(image_tensor):
    processed_tensor = (image_tensor - 127.5) / 127.5
    return processed_tensor

class GaussianNoise(object):
    def __init__(self, mean=0, std=0.1):
        self.mean = mean
        self.std = std

    def __call__(self, img):
        img_np = np.array(img, dtype=np.float32)
        row, col, ch = img_np.shape

        # Tính toán kích thước của vùng ảnh sẽ nhận nhiễu
        min_size = int(row * col * 0.1)
        max_size = int(row * col * 0.25)
        area_size = np.random.randint(min_size, max_size)

        # Tạo ma trận mask để chỉ định vùng của ảnh sẽ nhận nhiễu
        mask = np.zeros((row, col), dtype=np.uint8)
        x = np.random.randint(0, col)
        y = np.random.randint(0, row)
        x_end = min(x + int(np.sqrt(area_size)), col)
        y_end = min(y + int(np.sqrt(area_size)), row)
        mask[y:y_end, x:x_end] = 1

        # Tạo ma trận nhiễu Gaussian
        std = np.random.uniform(0, 0.1)
        gauss = np.random.normal(self.mean, std, (row, col, ch))

        # Áp dụng nhiễu vào phần của ảnh được chỉ định bởi mask
        noisy_img = img_np + gauss * mask[:, :, np.newaxis]
        noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)

        return Image.fromarray(noisy_img)

In [5]:
train_transform = transforms.Compose([
    transforms.RandomCrop(224),
    GaussianNoise(),
    transforms.RandomHorizontalFlip(p=0.3),
    np.float32,
    transforms.ToTensor(),
    fixed_image_standardization
])

test_transform = transforms.Compose([
    transforms.CenterCrop(224),
    np.float32,
    transforms.ToTensor(),
    fixed_image_standardization
])

Phải chuyển về bmp để PIL đọc huhu

In [6]:
train_data = datasets.ImageFolder(CONFIGURATION['train_dir'], transform=train_transform)
test_data = datasets.ImageFolder(CONFIGURATION['test_dir'], transform=test_transform)

# Đếm số lượng mẫu mỗi lớp
labels = [label for _, label in train_data]
class_counts = Counter(labels)

# Tính trọng số cho mỗi lớp (trọng số là tỉ lệ nghịch với số lượng mẫu)
weights = 1. / np.array([class_counts[i] for i in range(len(train_data.classes))])
sample_weights = [weights[label] for _, label in train_data]

# Tạo WeightedRandomSampler
sampler = WeightedRandomSampler(
    weights=sample_weights,      # Trọng số cho mỗi mẫu
    num_samples=len(sample_weights), # Tổng số mẫu
    replacement=True              # Cho phép trùng lặp mẫu trong mỗi epoch
)

# Turn images into data loaders
train_dataloader = DataLoader(
    train_data,
    batch_size=CONFIGURATION['batch_size'],
    sampler=sampler,
    # shuffle=True,
    num_workers=CONFIGURATION['num_workers'],
    pin_memory=True,
)
test_dataloader = DataLoader(
    test_data,
    batch_size=CONFIGURATION['batch_size'],
    shuffle=False, # don't need to shuffle test data
    num_workers=CONFIGURATION['num_workers'],
    pin_memory=True,
)

In [7]:
dataiter = iter(train_dataloader)
images, labels = next(dataiter)
print(f"Labels: {labels}")  # Expect tensor of integers

Labels: tensor([ 17,  64,  97,  90, 106, 164,  37, 134,  44,  97, 104,   4,  16,   1,
        128, 160])


# Utils

In [8]:
class EarlyStopping:
    def __init__(
        self, 
        monitor='val_accuracy', 
        min_delta=0, 
        patience=0, 
        verbose=0, 
        mode='auto', 
        baseline=None, 
        restore_best_weights=False, 
        start_from_epoch=0, 
        save_path=None
    ):
        """
        PyTorch EarlyStopping giống với tf.keras.callbacks.EarlyStopping
        
        Args:
            monitor (str): Metric để theo dõi, ví dụ 'val_loss'.
            min_delta (float): Sai số nhỏ nhất để tính là có cải thiện.
            patience (int): Số epoch không cải thiện trước khi dừng.
            verbose (int): Hiển thị log nếu > 0.
            mode (str): 'min', 'max' hoặc 'auto'.
            baseline (float): Giá trị ban đầu yêu cầu metric phải vượt qua.
            restore_best_weights (bool): Khôi phục trọng số tốt nhất khi dừng.
            start_from_epoch (int): Bắt đầu kiểm tra early stopping từ epoch này.
            save_path (str): Đường dẫn để lưu trọng số tốt nhất (ghi đè).
        """
        self.min_delta = min_delta
        self.patience = patience
        self.verbose = verbose
        self.mode = mode
        self.baseline = baseline
        self.restore_best_weights = restore_best_weights
        self.start_from_epoch = start_from_epoch
        self.save_path = save_path

        # Cấu hình chế độ
        if mode not in ['min', 'max', 'auto']:
            raise ValueError(f"Invalid mode '{mode}', must be 'min', 'max', or 'auto'.")
        if mode == 'min' or (mode == 'auto' and 'loss' in monitor):
            self.monitor_op = lambda current, best: current < best - min_delta
            self.best_value = float('inf')
        elif mode == 'max' or (mode == 'auto' and 'acc' in monitor):
            self.monitor_op = lambda current, best: current > best + min_delta
            self.best_value = -float('inf')

        self.counter = 0
        self.early_stop = False
        self.best_epoch = -1

    def __call__(self, current_value, model, epoch):
        # Không kiểm tra trước start_from_epoch
        if epoch < self.start_from_epoch:
            return
        
        # Kiểm tra baseline
        if self.baseline is not None and epoch == 0:
            if not self.monitor_op(current_value, self.baseline):
                if self.verbose:
                    cprint(f"Metric '{self.monitor}' did not meet baseline {self.baseline}. Early stopping enabled.", 'red')
                self.early_stop = True
                self.early_stop = True
                return

        # Kiểm tra cải thiện
        if self.monitor_op(current_value, self.best_value):
            self.best_value = current_value
            self.best_epoch = epoch
            self.counter = 0

            # Lưu trọng số tốt nhất vào file
            if self.save_path:
                torch.save(model.state_dict(), self.save_path)
                if self.verbose:
                    cprint(f"Saved best model weights at epoch {epoch} to '{self.save_path}'", 'green')
        else:
            self.counter += 1
            if self.verbose:
                cprint(f"Epoch {epoch}: EarlyStopping counter: {self.counter}/{self.patience}", 'light_red')
            if self.counter >= self.patience:
                if self.verbose:
                    cprint(f"Early stopping triggered at epoch {epoch}.", 'red')
                self.early_stop = True

                # Khôi phục trọng số từ file
                if self.restore_best_weights and self.save_path:
                    model.load_state_dict(torch.load(self.save_path))
                    if self.verbose:
                        cprint(f"Restored best model weights from '{self.save_path}'", 'blue')

In [9]:
class ModelCheckpoint:
    def __init__(self, filepath, verbose=0):
        self.filepath = filepath
        self.verbose = verbose

    def __call__(self, model, epoch):
        # Lưu model vào file với tên cố định mỗi epoch
        torch.save(model.state_dict(), self.filepath)
        if self.verbose > 0:
            cprint(f"Saving model to {self.filepath}", 'cyan')

In [10]:
class MagLoss(torch.nn.Module):
    def __init__(self, l_a=CONFIGURATION['l_a'], u_a=CONFIGURATION['u_a'], lambda_g=CONFIGURATION['lambda_g']):
        super(MagLoss, self).__init__()
        self.l_a = l_a
        self.u_a = u_a
        self.lambda_g = lambda_g

    def calc_loss_G(self, x_norm):
        g = 1/(self.u_a**2) * x_norm + 1/(x_norm)
        # x_norm là 1 tensor chứa batch các ai. Do đó g là 1 tensor chứa batch các giá trị g thỏa mãn cho mỗi sample.
        # Việc lấy giá trị trung bình giúp làm mượt quá trình huấn luyện (nếu không loss thay đổi rất nhanh, vì mỗi mẫu trong batch có thể có độ khó khác nhau). Khi tính trung bình, các giá trị sẽ được làm mượt, giúp quá trình huấn luyện diễn ra mượt mà hơn và không bị ảnh hưởng quá nhiều bởi các mẫu cực trị.
        # Đảm bảo rằng loss không thay đổi theo kích thước batch. Điều này rất quan trọng trong việc huấn luyện mô hình một cách công bằng giữa các batch có kích thước khác nhau.
        return torch.mean(g)

    # input: là logits thu được của layer MagLinear.
    # target: là tensor label thực tế của cả batch
    # x_norm: là logits thu được của layer MagLinear
    def forward(self, input, target, x_norm):
        loss_g = self.calc_loss_G(x_norm)

        cos_theta, cos_theta_m = input
        
        # Khởi tạo 1 tensor chứa các giá trị 0 có kích thước giống như cos_theta
        one_hot = torch.zeros_like(cos_theta)
        # Onehotcoding label. Phương thức scatter_ sẽ thay các giá trị của one_hot tại chỉ mục được xác định bởi target thành 1.
        # 1: dim=1, Chỉ thị cho PyTorch biết sẽ thay đổi giá trị dọc theo chiều thứ nhất (theo các cột, tức là các lớp).
        # target.view(-1, 1): chuyển target thành tensor cột có kích thước (batch_size,1)
        # 1.0: giá trị được gán
        one_hot.scatter_(1, target.view(-1, 1), 1.0)
        
        # Onehot là ma trận mục tiêu, ta muốn áp dụng cos_theta_m cho lớp mục tiêu này và giữ nguyên giá trị cosin_theta cho các lớp khác.
        # Đây là cách thức áp dụng margin vào cosine similarity để tăng cường phân biệt giữa lớp mục tiêu và các lớp khác.
        output = one_hot * cos_theta_m + (1.0 - one_hot) * cos_theta
        loss_c = F.cross_entropy(output, target, reduction='mean')
        # Chỉ trả về các thành phần tính loss chứ chưa thực sự tính
        return loss_c + self.lambda_g * loss_g

In [11]:
class EpochAccuracy:
    def __init__(self):
        self.correct_top1 = 0
        self.correct_top5 = 0
        self.total_samples = 0

    # predict là batch vector softmax của batch sample.
    def update(self, predict, target):
        with torch.no_grad():
            batch_size = target.size(0)

            # Lấy top-5 dự đoán có cos(theta_m) lớn nhất
            _, pred = predict.topk(5, 1, True, True)
            
            pred = pred.t()

            # Đảm bảo target là một tensor chứa các chỉ số lớp, không phải one-hot encoding
            correct = pred.eq(target.view(1, -1).expand_as(pred))
            
            self.correct_top1 += correct[:1].contiguous().view(-1).float().sum(0, keepdim=True).item()
            self.correct_top5 += correct[:5].contiguous().view(-1).float().sum(0, keepdim=True).item()
            self.total_samples += batch_size

    def compute(self):
        if self.total_samples == 0:
            return 0
        top_1 = (self.correct_top1 / self.total_samples) * 100
        top_5 = (self.correct_top5 / self.total_samples) * 100
        return round(top_1, 3), round(top_5, 3)

    def reset(self):
        self.correct_top1 = 0
        self.correct_top5 = 0
        self.total_samples = 0

2 Chiến lược điều chỉnh learning rate tốt nhất là Stochastic Gradient Descent with Restarts (SGDR) và 1cycle (Super-Convergence)

# Model Architech

## Tmp

In [12]:
def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)


def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, norm_layer=None,
                 start_block=False, end_block=False, exclude_bn0=False):
        super(BasicBlock, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        # Both self.conv1 and self.downsample layers downsample the input when stride != 1
        if not start_block and not exclude_bn0:
            self.bn0 = norm_layer(inplanes)

        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)

        if start_block:
            self.bn2 = norm_layer(planes)

        if end_block:
            self.bn2 = norm_layer(planes)

        self.downsample = downsample
        self.stride = stride

        self.start_block = start_block
        self.end_block = end_block
        self.exclude_bn0 = exclude_bn0

    def forward(self, x):
        identity = x

        if self.start_block:
            out = self.conv1(x)
        elif self.exclude_bn0:
            out = self.relu(x)
            out = self.conv1(out)
        else:
            out = self.bn0(x)
            out = self.relu(out)
            out = self.conv1(out)

        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)

        if self.start_block:
            out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity

        if self.end_block:
            out = self.bn2(out)
            out = self.relu(out)

        return out

class iResNet(nn.Module):

    def __init__(self, block, layers, num_classes=1000, zero_init_residual=False, norm_layer=None, dropout_prob0=0.0):
        super(iResNet, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3,
                               bias=False)
        self.bn1 = norm_layer(64)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(block, 64, layers[0], stride=2, norm_layer=norm_layer)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, norm_layer=norm_layer)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, norm_layer=norm_layer)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, norm_layer=norm_layer)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        if dropout_prob0 > 0.0:
            self.dp = nn.Dropout(dropout_prob0, inplace=True)
            print("Using Dropout with the prob to set to 0 of: ", dropout_prob0)
        else:
            self.dp = None

        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1, norm_layer=None):
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        downsample = None
        if stride != 1 and self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.MaxPool2d(kernel_size=3, stride=stride, padding=1),
                conv1x1(self.inplanes, planes * block.expansion),
                norm_layer(planes * block.expansion),
            )
        elif self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion),
                norm_layer(planes * block.expansion),
            )
        elif stride != 1:
            downsample = nn.MaxPool2d(kernel_size=3, stride=stride, padding=1)

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, norm_layer,
                            start_block=True))
        self.inplanes = planes * block.expansion
        exclude_bn0 = True
        for _ in range(1, (blocks-1)):
            layers.append(block(self.inplanes, planes, norm_layer=norm_layer,
                                exclude_bn0=exclude_bn0))
            exclude_bn0 = False

        layers.append(block(self.inplanes, planes, norm_layer=norm_layer, end_block=True, exclude_bn0=exclude_bn0))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)

        if self.dp is not None:
            x = self.dp(x)

        x = self.fc(x)

        return x


def iresnet34(**kwargs):
    """Constructs a iResNet-18 model.

    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    return iResNet(BasicBlock, [3, 4, 6, 3], **kwargs)

In [13]:
backbone = iresnet34(num_classes=512)

In [14]:
summary(model=backbone, 
        input_size=(CONFIGURATION['batch_size'], 3, CONFIGURATION['image_size'], CONFIGURATION['image_size']),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
) 

Layer (type (var_name))                  Input Shape          Output Shape         Param #              Trainable
iResNet (iResNet)                        [16, 3, 224, 224]    [16, 512]            --                   True
├─Conv2d (conv1)                         [16, 3, 224, 224]    [16, 64, 112, 112]   9,408                True
├─BatchNorm2d (bn1)                      [16, 64, 112, 112]   [16, 64, 112, 112]   128                  True
├─ReLU (relu)                            [16, 64, 112, 112]   [16, 64, 112, 112]   --                   --
├─Sequential (layer1)                    [16, 64, 112, 112]   [16, 64, 56, 56]     --                   True
│    └─BasicBlock (0)                    [16, 64, 112, 112]   [16, 64, 56, 56]     --                   True
│    │    └─Conv2d (conv1)               [16, 64, 112, 112]   [16, 64, 56, 56]     36,864               True
│    │    └─BatchNorm2d (bn1)            [16, 64, 56, 56]     [16, 64, 56, 56]     128                  True
│    │    └─ReLU

## code

In [15]:
class BackBone(torch.nn.Module):
    def __init__(self, out_features):
        super(BackBone, self).__init__()
        # weights = models.EfficientNet_B0_Weights.DEFAULT
        # self.backbone = models.efficientnet_b0(weights=weights)
        self.backbone = models.efficientnet_b0()
        self.backbone.classifier = torch.nn.Sequential(
            torch.nn.Dropout(p=0.2, inplace=True), 
            torch.nn.Linear(in_features=1280, out_features=out_features, bias=True)
        )
    def forward(self, x):
        return self.backbone(x)

In [16]:
summary(model=BackBone(512), 
        input_size=(CONFIGURATION['batch_size'], 3, CONFIGURATION['image_size'], CONFIGURATION['image_size']),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

Layer (type (var_name))                                           Input Shape          Output Shape         Param #              Trainable
BackBone (BackBone)                                               [16, 3, 224, 224]    [16, 512]            --                   True
├─EfficientNet (backbone)                                         [16, 3, 224, 224]    [16, 512]            --                   True
│    └─Sequential (features)                                      [16, 3, 224, 224]    [16, 1280, 7, 7]     --                   True
│    │    └─Conv2dNormActivation (0)                              [16, 3, 224, 224]    [16, 32, 112, 112]   928                  True
│    │    └─Sequential (1)                                        [16, 32, 112, 112]   [16, 16, 112, 112]   1,448                True
│    │    └─Sequential (2)                                        [16, 16, 112, 112]   [16, 24, 56, 56]     16,714               True
│    │    └─Sequential (3)                               

In [17]:
class MagLinear(torch.nn.Module):
    def __init__(self, in_features, out_features, scale = 64.0, easy_margin=True, l_margin=0.45, u_margin=0.8, l_a=10, u_a=110):
        super(MagLinear, self).__init__()
        # Tạo ra ma trận cần train của lớp FullyConnected này
        self.weight = Parameter(torch.Tensor(in_features, out_features))
        # Can thiệp vào quá trình khởi tạo trọng số: 
        # uniform_(-1,1): đảm bảo phân phối đồng đều (uniform) trong khoảng [-1,1]
        # renorm_(2, 1, 1e-5): chuẩn hóa lại các giá trị theo 1 chiều cụ thể. 
        #   2: Chuẩn hóa theo chuẩn L2(Euclidean norm).
        #   1: Thực hiện chuẩn hóa trên chiều thứ nhất của tensor (các hàng của ma trận self.weight)
        #   1e-5: Giá trị ngưỡng (epsilon) để tránh chia cho số 0
        # Khi dùng chuẩn hóa này, mỗi hàng của ma trận trọng số sẽ được điều chỉnh sao cho norm của chúng không vượt quá một giá trị cụ thể (giới hạn bởi epsilon).
        # Bản thân mỗi cột của ma trận này là ma trận weight của 1 neutron hay vector tâm của class identity.
        self.weight.data.uniform_(-1,1).renorm_(2, 1, 1e-5).mul_(1e5)
        
        self.scale, self.easy_margin = scale, easy_margin
        self.l_margin, self.u_margin = l_margin, u_margin
        self.l_a, self.u_a = l_a, u_a
    
    # Generate adaptive margin
    def _margin(self, x):
        margin = (self.u_margin-self.l_margin) / \
            (self.u_a-self.l_a)*(x-self.l_a) + self.l_margin
        return margin
    
    # x là batch_embedding
    def forward(self, x):
        # Tính độ dài của batch embedding sinh ra batch ai
        # Dim = 1: Tính độ dài theo chiều batch của tensor vào. Keepdim=True đảm bảo tensor sau tính toán sẽ có shape (batch_size,1)
        # clamp(min, max) giới hạn giá trị trong tensor sao cho tất cả các phần tử nhỏ hơn min sẽ được thay thế bằng min, và tất cả các phần tử lớn hơn max sẽ được thay thế bằng max.
        x_norm = torch.norm(x, dim=1, keepdim=True).clamp(self.l_a, self.u_a)
        
        # ada chắc là viết tắt của adaptive
        ada_margin = self._margin(x_norm)
        
        cos_m, sin_m = torch.cos(ada_margin), torch.sin(ada_margin)
        
        # Tính theta yi
        # dim bằng 0 tức là chuẩn hóa ma trận theo cột => mỗi cột là ma trận trọng số của 1 neutron như đã nói và sẽ có độ dài là 1
        weight_norm = F.normalize(self.weight, dim=0)
        # cosine similarity (sự tương đồng cosin). Kết quả thu được tensor chứa batch_cosin_theta
        cos_theta = torch.mm(F.normalize(x), weight_norm)
        # Đảm bảo giá trị cosin của các vector không vượt quá giới hạn => Thừa
        cos_theta = cos_theta.clamp(-1, 1)
        # sin sẽ luôn dương
        sin_theta = torch.sqrt(1.0 - torch.pow(cos_theta, 2))
        # cos(a+b) = cos(a)*cos(b) - sin(a)*sin(b)
        cos_theta_m = cos_theta * cos_m - sin_theta * sin_m

        # easy_margin = True để điều chỉnh cos_theta_m, nó sẽ bằng công thức trên nếu cos_theta>0 và không điều chỉnh gì nếu cos_theta<0
        # Cách này làm đơn giản việc tính toán
        # Nếu muốn điều chỉnh thì cần thêm vào threshold nào đó (code của họ để mặc định easy_margin=False)
        if self.easy_margin:
            cos_theta_m = torch.where(cos_theta > 0, cos_theta_m, cos_theta)
        else:
            mm = torch.sin(math.pi - ada_margin) * ada_margin
            threshold = torch.cos(math.pi - ada_margin)
            cos_theta_m = torch.where(cos_theta > threshold, cos_theta_m, cos_theta - mm)
            
        # multiply the scale in advance
        cos_theta_m = self.scale * cos_theta_m
        cos_theta = self.scale * cos_theta

        # Trả về các giá trị này để tính loss và accuracy
        # [cos_theta, cos_theta_m] là logits của lớp này. Lý do thêm x_norm để phục vụ MagFace+ => Train không ?
        # cos_theta chính là accuracy, nó đo cosine similarity giữa vector tâm và feature vector.
        return [cos_theta, cos_theta_m], x_norm

In [18]:
class FaceRecognition(torch.nn.Module):
    def __init__(self):
        super(FaceRecognition, self).__init__()
        self.feature_extractor = BackBone(CONFIGURATION['embedding_size'])
        self.classification = MagLinear(CONFIGURATION['embedding_size'], CONFIGURATION['num_class'])
    
    def forward(self, x):
        x = self.feature_extractor(x)
        logits, x_norm = self.classification(x)
        
        return logits, x_norm

In [19]:
# Define the dummy target with the shape [batch_size, num_class]
dummy_target = torch.zeros(CONFIGURATION['batch_size'], CONFIGURATION['num_class'])

# Define input data tensor with the shape [batch_size, 3, image_size, image_size]
input_data = torch.randn(CONFIGURATION['batch_size'], 3, CONFIGURATION['image_size'], CONFIGURATION['image_size'])

# Call summary with both input_data and dummy_target
summary(model=FaceRecognition(), 
        input_data=(input_data),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                                Input Shape          Output Shape         Param #              Trainable
FaceRecognition (FaceRecognition)                                      [16, 3, 224, 224]    [16, 203]            --                   True
├─BackBone (feature_extractor)                                         [16, 3, 224, 224]    [16, 512]            --                   True
│    └─EfficientNet (backbone)                                         [16, 3, 224, 224]    [16, 512]            --                   True
│    │    └─Sequential (features)                                      [16, 3, 224, 224]    [16, 1280, 7, 7]     4,007,548            True
│    │    └─AdaptiveAvgPool2d (avgpool)                                [16, 1280, 7, 7]     [16, 1280, 1, 1]     --                   --
│    │    └─Sequential (classifier)                                    [16, 1280]           [16, 512]            655,872              True
├─MagLinear (classificat

# Train loop

In [20]:
model = FaceRecognition().to(device)

In [21]:
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIGURATION['learning_rate'])

In [22]:
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=1, eta_min=1e-6)

In [23]:
accuracy = EpochAccuracy()

criterion = MagLoss()

modle_chekpoint = ModelCheckpoint(filepath='checkpoint/efficient_b0/checkpoint.pth', verbose=1)

early_stopping = EarlyStopping(monitor='val_accuracy', patience=200, restore_best_weights=True, save_path='checkpoint/efficient_b0/best_weight.pth', verbose=1, start_from_epoch=50)

In [24]:
def train_step(train_dataloader, model, device, accuracy, optimizer):
    train_loss = 0
    
    model.train()
    for (X, y) in train_dataloader:
        X, y = X.to(device), y.to(device)
        
        # 1. Forward pass
        logits, x_norm = model(X)
        
        # 2. Accuracy top 1 anh top 5
        accuracy.update(logits[0], y)
        
        # 3. Calculate  and accumulate loss
        train_loss = criterion(logits, y, x_norm)
        train_loss += train_loss
        
        # 4. Optimizer zero grad
        optimizer.zero_grad()

        # 5. Loss backward
        train_loss.backward()

        # 6. Optimizer step
        optimizer.step()
    
    train_accuracy_top_1, train_accuracy_top_5 = accuracy.compute()
    print(f"train_loss: {train_loss/len(train_dataloader)} - train_accuracy_top_1: {train_accuracy_top_1}% - train_accuracy_top_5: {train_accuracy_top_5}%")
    accuracy.reset()

In [25]:
def test_step(test_dataloader, model, device, accuracy):
    model.eval()
    test_loss = 0
    with torch.inference_mode():
        for (X, y) in test_dataloader:
            X, y = X.to(device), y.to(device)
            logis, x_norm = model(X)
            test_loss = criterion(logis, y, x_norm)
            test_loss += test_loss
            accuracy.update(logis[0], y)
    
    test_accuracy_top_1, test_accuracy_top_5 = accuracy.compute()
    print(f"val_loss: {test_loss/len(test_dataloader)} - val_accuracy_top_1: {test_accuracy_top_1}% - val_accuracy_top_5: {test_accuracy_top_5}%")
    
    accuracy.reset()
    return test_loss

In [26]:
for epoch in range(CONFIGURATION['epochs']):
    cprint(f"Epoch: {epoch+1}\n-------", 'cyan')
    
    train_step(train_dataloader, model, device, accuracy, optimizer)
    
    # Validate
    test_loss = test_step(test_dataloader, model, device, accuracy)
    
    modle_chekpoint(model, epoch+1)
    early_stopping(test_loss, model, epoch+1)
    if early_stopping.early_stop:
        break
    
    # Điều chỉnh learning rate:
    scheduler.step()

Epoch: 1
-------
train_loss: 0.21668320894241333 - train_accuracy_top_1: 0.526% - train_accuracy_top_5: 2.477%
val_loss: 2.027299642562866 - val_accuracy_top_1: 0.0% - val_accuracy_top_5: 1.111%
Saving model to checkpoint/efficient_b0/checkpoint.pth
Epoch: 2
-------
train_loss: 0.16615904867649078 - train_accuracy_top_1: 0.601% - train_accuracy_top_5: 3.228%
val_loss: 2.1600985527038574 - val_accuracy_top_1: 1.111% - val_accuracy_top_5: 2.963%
Saving model to checkpoint/efficient_b0/checkpoint.pth
Epoch: 3
-------
train_loss: 0.13794471323490143 - train_accuracy_top_1: 0.45% - train_accuracy_top_5: 3.453%
val_loss: 0.7264842987060547 - val_accuracy_top_1: 0.0% - val_accuracy_top_5: 0.37%
Saving model to checkpoint/efficient_b0/checkpoint.pth
Epoch: 4
-------
train_loss: 0.149506613612175 - train_accuracy_top_1: 0.601% - train_accuracy_top_5: 3.453%
val_loss: 0.8849567770957947 - val_accuracy_top_1: 0.741% - val_accuracy_top_5: 6.667%
Saving model to checkpoint/efficient_b0/checkpoint.p

KeyboardInterrupt: 

À quên: còn vấn đề mất cân bằng dữ liệu và data augumentation

In [ ]:
torch.cuda.empty_cache()